# ParkirIn: Hitung Slot Kosong Parkiran Motor (YOLOv8)

Notebook ini adalah versi rapi dari seluruh eksperimen. Isinya hanya alur yang akhirnya dipakai: **model fine-tune `motor_depan_v3`** (`conf=0.45`, `iou=0.6`, `imgsz=960`) dan hasil akhirnya `hasil_v3.csv` untuk dashboard Streamlit.

## Alur

| Bagian | Isi | Manual? |
|---|---|---|
| 0 | Setup, konfigurasi, mount Drive | - |
| 1 | Baca dataset (PAGI, MALAM), parse nama file | - |
| 2 | Kapasitas slot per blok | **isi `kapasitas.csv`** |
| 3 | Baseline pretrained + filter (pembanding) | - |
| 4 | Siapkan dataset fine-tune (pra-label, Roboflow, susun) | **koreksi label di Roboflow** |
| 5 | Training `motor_depan_v3` | - |
| 6 | Evaluasi: val ROW6, test ROW7-8 vs baseline | **isi `gt_test.csv`** |
| 7 | Output akhir `hasil_v3.csv` | - |

## Ringkasan hasil

| Pendekatan | Hasil |
|---|---|
| Baseline pretrained + filter (`conf=0.5, area=0.02, y_min=0.65`) | MAE ~0.55 di 30 foto tuning, **1.00** di ROW7-8 |
| Fine-tune v3 (val ROW6) | mAP50 0.977, precision 0.955, recall 0.929 |
| Fine-tune v3 (`conf` 0.4-0.5 di val) | MAE 0.50, bias mendekati 0 |
| Fine-tune v3 (uji ROW7-8, 16 foto) | MAE **0.81** (pagi 0.62, malam 1.00) |

**Batasan:** akurasi sekitar ±1 motor per foto. Lemah di blok **padat malam** (contoh ROW7A, ROW7B). Laporkan hasil per blok sebagai status (penuh / sebagian / banyak kosong), bukan angka pasti.

**Aturan label/hitung:** hanya motor barisan depan, termasuk yang terpotong di tepi (kelihatan minimal ~40%), tanpa pengendara. Motor barisan belakang tidak dilabeli.

## 0. Setup

In [ ]:
!pip install -q ultralytics opencv-python-headless pandas tqdm matplotlib

In [ ]:
import os, re, glob, shutil, random
from datetime import datetime
from zoneinfo import ZoneInfo

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from ultralytics import YOLO
from google.colab import drive, files

drive.mount('/content/drive')

In [ ]:
# ---------------- Konfigurasi (ubah di sini saja) ----------------
DATA_DIR   = '/content/drive/MyDrive/PARKIR DATASET'   # berisi folder PAGI dan MALAM
OUT        = '/content/hasil'                          # folder output sementara
RUN_DIR    = '/content/drive/MyDrive/yolo_parkir'      # tempat hasil training
SESI       = ['PAGI', 'MALAM']

MODEL_NAME = 'motor_depan_v3'
BEST_PT    = f'{RUN_DIR}/{MODEL_NAME}/weights/best.pt'
RETRAIN    = False        # True = latih ulang walau best.pt sudah ada

# Parameter inference model fine-tune (hasil tuning di val ROW6)
CONF_FT, IOU, IMGSZ = 0.45, 0.6, 960

# Parameter baseline pretrained + filter (hasil tuning di 30 foto)
C_BASE, A_BASE, Y_BASE = 0.5, 0.02, 0.65

ZIP_ROBOFLOW = 'motor_depan_v2.v1i.yolov8.zip'          # nama zip export Roboflow

os.makedirs(OUT, exist_ok=True)

import torch
print('GPU aktif:', torch.cuda.is_available())          # harus True (Runtime > T4 GPU)

## 1. Baca dataset

Struktur: `DATA_DIR/PAGI/*.jpg` dan `DATA_DIR/MALAM/*.jpg`. Nama file `ROW{n}{A-F}_{SESI}.jpg` berarti baris parkir ke-n, blok A-F. Foto `SIDE*` dan nama bawaan HP dipisah dan tidak ikut perhitungan slot.

In [ ]:
def parse(nama):
    m = re.match(r'(ROW\d+)([A-Z]?)_', nama, re.IGNORECASE)
    if m:
        return 'ROW', m.group(1).upper(), m.group(2).upper()
    if nama.upper().startswith('SIDE'):
        return 'SIDE', 'SIDE', nama[4:5].upper()
    return 'LAIN', 'LAIN', ''

rows = []
for sesi in SESI:
    for f in sorted(glob.glob(os.path.join(DATA_DIR, sesi, '*.jpg'))):
        nama = os.path.basename(f)
        tipe, row, titik = parse(nama)
        rows.append(dict(sesi=sesi, file=nama, tipe=tipe, row=row, titik=titik))

df = pd.DataFrame(rows)
print(df.groupby(['sesi', 'tipe']).size().unstack(fill_value=0))

## 2. Kapasitas slot per blok

Kapasitas dihitung manual **sekali per blok** (cukup dari foto pagi), karena titik pengambilan sama di pagi dan malam. Rumus: `slot_kosong = kapasitas - motor_depan`.

> Catatan: kapasitas **ROW1A** sebelumnya terdeteksi kurang (hitungan manual 5-6, kapasitas terisi 4). Periksa dan koreksi di `kapasitas.csv`.

In [ ]:
# Buat template kapasitas, lalu isi kolom 'kapasitas' secara manual
cap_tpl = (df[df.tipe == 'ROW'][['row', 'titik']]
           .drop_duplicates().sort_values(['row', 'titik']))
cap_tpl['kapasitas'] = ''
cap_tpl.to_csv(f'{OUT}/kapasitas.csv', index=False)
files.download(f'{OUT}/kapasitas.csv')
print(len(cap_tpl), 'blok')

In [ ]:
# Upload kapasitas.csv yang sudah diisi
up = files.upload()
cap = pd.read_csv('kapasitas.csv')
assert cap['kapasitas'].notna().all(), 'Masih ada kapasitas yang kosong'

d = df[df.tipe == 'ROW'].merge(cap, on=['row', 'titik'])
d['kapasitas'] = d['kapasitas'].astype(int)
print(d.shape, '| kapasitas per blok:', d['kapasitas'].min(), '-', d['kapasitas'].max())

## 3. Baseline: YOLOv8m pretrained + filter

Dipakai sebagai **pembanding** model fine-tune. Deteksi kelas `motorcycle` (COCO id 3), lalu disaring dengan `conf`, ukuran box (`area`), dan posisi bawah box (`y_min`). Parameter `0.5 / 0.02 / 0.65` dipilih dari grid di 30 foto ground truth (area datar, bukan titik terbaik tunggal).

In [ ]:
model_base = YOLO('yolov8m.pt')

# Simpan deteksi mentah sekali (conf rendah), filter dilakukan di CPU
raw = {}
for sesi in SESI:
    for f in tqdm(sorted(glob.glob(os.path.join(DATA_DIR, sesi, '*.jpg'))), desc=sesi):
        img = cv2.imread(f); h, w = img.shape[:2]
        r = model_base(img, classes=[3], conf=0.25, iou=0.6, verbose=False)[0]
        raw[(sesi, os.path.basename(f))] = dict(
            w=w, h=h,
            boxes=r.boxes.xyxy.cpu().numpy(),
            conf=r.boxes.conf.cpu().numpy())

def hitung_base(key, conf_t=C_BASE, area_t=A_BASE, y_t=Y_BASE):
    r = raw[key]; n = 0
    for (x1, y1, x2, y2), c in zip(r['boxes'], r['conf']):
        area = (x2 - x1) * (y2 - y1) / (r['w'] * r['h'])
        if c >= conf_t and area >= area_t and y2 >= y_t * r['h']:
            n += 1
    return n

d['motor_depan_base'] = [hitung_base((r.sesi, r.file)) for r in d.itertuples()]
print('Foto dengan terdeteksi > kapasitas:', (d['motor_depan_base'] > d['kapasitas']).sum())

In [ ]:
# Ground truth sampel (ROW1-6 saja; ROW7-8 disimpan untuk uji akhir).
# Isi kolom gt_depan manual dengan aturan label yang sama, lalu upload lagi.
sample = (d[d.row.isin([f'ROW{i}' for i in range(1, 7)])]
          .groupby('sesi').sample(15, random_state=42)[['sesi', 'file']])
sample['gt_depan'] = ''
sample.to_csv('gt_sample.csv', index=False)
files.download('gt_sample.csv')

In [ ]:
# Evaluasi baseline di ground truth sampel (jalankan setelah gt_sample.csv terisi)
up = files.upload()
gt = pd.read_csv('gt_sample.csv')
gt['gt_depan'] = gt['gt_depan'].astype(float)
gt['pred'] = [hitung_base((r.sesi, r.file)) for r in gt.itertuples()]
gt['err'] = gt['pred'] - gt['gt_depan']

print('MAE:', gt['err'].abs().mean().round(2), '| bias:', gt['err'].mean().round(2))
print(gt.groupby('sesi')['err'].agg(MAE=lambda x: x.abs().mean(), bias='mean').round(2))

## 4. Dataset fine-tune (satu kelas: `motor_depan`)

Alur: pra-label otomatis dari model pretrained, lalu **koreksi manual di Roboflow** (hapus motor belakang, tambah motor depan yang terlewat), export YOLOv8, lalu disusun per ROW.

Pembagian per ROW (bukan acak per foto, supaya tidak bocor): **ROW1-5 = train, ROW6 = val, ROW7-8 = test (tidak dilabeli)**.

### 4.1 Pra-label (ROW1-6)

In [ ]:
PRE = '/content/prelabel'
shutil.rmtree(PRE, ignore_errors=True)
os.makedirs(f'{PRE}/images', exist_ok=True)
os.makedirs(f'{PRE}/labels', exist_ok=True)

def row_no(nama):
    m = re.match(r'ROW(\d+)', nama, re.IGNORECASE)
    return int(m.group(1)) if m else None

jumlah = 0
for sesi in SESI:
    for f in sorted(glob.glob(os.path.join(DATA_DIR, sesi, '*.jpg'))):
        nama = os.path.basename(f)
        n = row_no(nama)
        if n is None or n > 6:
            continue
        new = f'{sesi}_{nama}'
        shutil.copy(f, f'{PRE}/images/{new}')
        r = model_base(f, classes=[3], conf=0.4, iou=0.6, verbose=False)[0]
        with open(f'{PRE}/labels/{new[:-4]}.txt', 'w') as fh:
            for x, y, w_, h_ in r.boxes.xywhn.cpu().numpy():
                fh.write(f'0 {x:.6f} {y:.6f} {w_:.6f} {h_:.6f}\n')
        jumlah += 1

with open(f'{PRE}/data.yaml', 'w') as fh:       # agar Roboflow mengenali nama kelas
    fh.write('names:\n  0: motor_depan\nnc: 1\n')

print('Foto diproses:', jumlah)                  # seharusnya 72
shutil.make_archive('/content/prelabel', 'zip', PRE)
files.download('/content/prelabel.zip')

### 4.2 Koreksi di Roboflow (manual)

1. Buat project **Object Detection**, upload `prelabel.zip`.
2. Tiap foto: hapus kotak motor barisan belakang, tambah kotak motor depan yang terlewat, rapikan kotak.
3. **Generate** versi baru dengan **Preprocessing dan Augmentation dimatikan** (Resize ikut dimatikan, kalau tidak gambar ter-stretch).
4. Download format **YOLOv8** (zip), lalu lanjut ke sel berikut.

### 4.3 Import hasil export

In [ ]:
up = files.upload()                    # pilih zip dari Roboflow
print(list(up.keys()))

In [ ]:
!unzip -q "{ZIP_ROBOFLOW}" -d /content/export_roboflow
!ls /content/export_roboflow

In [ ]:
# Gabungkan semua split bawaan Roboflow (split dari Roboflow tidak dipakai)
EXT, OUT_L = '/content/export_roboflow', '/content/labeled'
shutil.rmtree(OUT_L, ignore_errors=True)
os.makedirs(f'{OUT_L}/images', exist_ok=True)
os.makedirs(f'{OUT_L}/labels', exist_ok=True)

for s in ['train', 'valid', 'test']:
    for img in glob.glob(f'{EXT}/{s}/images/*.jpg'):
        shutil.copy(img, f'{OUT_L}/images/{os.path.basename(img)}')
    for lbl in glob.glob(f'{EXT}/{s}/labels/*.txt'):
        shutil.copy(lbl, f'{OUT_L}/labels/{os.path.basename(lbl)}')

print(len(os.listdir(f'{OUT_L}/images')), 'gambar |', len(os.listdir(f'{OUT_L}/labels')), 'label')

### 4.4 Susun dataset per ROW (pakai foto asli)

Gambar export Roboflow bisa ter-stretch jadi persegi. Karena label YOLO ternormalisasi (0-1), label tetap valid untuk foto asli, jadi **label dari Roboflow dipakai, gambarnya diambil dari `DATA_DIR`**. Nama file dibersihkan dari akhiran `_jpg.rf.xxxx`.

In [ ]:
DS = '/content/dataset'
shutil.rmtree(DS, ignore_errors=True)
for s in ['train', 'val']:
    os.makedirs(f'{DS}/images/{s}', exist_ok=True)
    os.makedirs(f'{DS}/labels/{s}', exist_ok=True)

def split_of(nama):
    n = int(re.search(r'ROW(\d+)', nama, re.IGNORECASE).group(1))
    return 'train' if n <= 5 else 'val'          # ROW1-5 train, ROW6 val

ok, hilang = 0, []
for lbl in glob.glob(f'{OUT_L}/labels/*.txt'):
    stem = os.path.basename(lbl)[:-4]
    bersih = re.sub(r'_jpg\.rf\..*$', '', stem)      # contoh: PAGI_ROW3D_PAGI
    sesi, rest = bersih.split('_', 1)                # PAGI, ROW3D_PAGI
    asli = os.path.join(DATA_DIR, sesi, rest + '.jpg')
    if not os.path.exists(asli):
        hilang.append(asli); continue
    s = split_of(bersih)
    shutil.copy(asli, f'{DS}/images/{s}/{bersih}.jpg')
    shutil.copy(lbl, f'{DS}/labels/{s}/{bersih}.txt')
    ok += 1

print('berhasil:', ok, '| tidak ketemu:', len(hilang), hilang[:3])
for s in ['train', 'val']:
    print(s, len(os.listdir(f'{DS}/images/{s}')), 'gambar |', len(os.listdir(f'{DS}/labels/{s}')), 'label')

open(f'{DS}/data.yaml', 'w').write(f'''path: {DS}
train: images/train
val: images/val
names:
  0: motor_depan
''')

In [ ]:
# Validasi format label: harus 5 angka per baris dan nilai 0-1
bad = []
for p in glob.glob(f'{DS}/labels/*/*.txt'):
    for i, line in enumerate(open(p)):
        v = line.split()
        if len(v) != 5 or not all(0 <= float(x) <= 1 for x in v[1:]):
            bad.append((os.path.basename(p), i, line.strip()))
print('label bermasalah:', len(bad), bad[:5])          # harus 0

In [ ]:
# Cek visual: kotak label harus menempel di motor barisan depan
imgs = random.sample(glob.glob(f'{DS}/images/train/*.jpg'), 4)
fig, axs = plt.subplots(1, 4, figsize=(28, 7))
for ax, p in zip(axs, imgs):
    im = cv2.imread(p); h, w = im.shape[:2]
    lbl = p.replace('images', 'labels')[:-4] + '.txt'
    for line in open(lbl):
        c, x, y, bw, bh = map(float, line.split()[:5])
        cv2.rectangle(im, (int((x - bw/2)*w), int((y - bh/2)*h)),
                          (int((x + bw/2)*w), int((y + bh/2)*h)), (0, 255, 0), 4)
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.set_title(os.path.basename(p)[:25]); ax.axis('off')
plt.show()

## 5. Training `motor_depan_v3`

Setelan akhir yang berhasil: backbone dibekukan (`freeze=10`), learning rate lebih rendah (`lr0=0.005`), dan mosaic dimatikan di 15 epoch terakhir. Dua percobaan sebelumnya dibuang karena tidak stabil: v1 (gambar ter-stretch) dan v2 (`freeze=0`, skor model berisik di data sekecil ini).

In [ ]:
if RETRAIN or not os.path.exists(BEST_PT):
    YOLO('yolov8m.pt').train(
        data=f'{DS}/data.yaml',
        epochs=100, imgsz=IMGSZ, batch=8, patience=0,
        freeze=10, lr0=0.005,
        hsv_v=0.5, degrees=3,          # variasi brightness (pagi vs malam), sedikit rotasi
        close_mosaic=15,
        project=RUN_DIR, name=MODEL_NAME,
    )
else:
    print('best.pt sudah ada, training dilewati:', BEST_PT)

best = YOLO(BEST_PT)

In [ ]:
# Kurva training: loss harus turun, mAP50 mendatar di akhir
res_csv = f'{RUN_DIR}/{MODEL_NAME}/results.csv'
if os.path.exists(res_csv):
    r = pd.read_csv(res_csv); r.columns = r.columns.str.strip()
    print(len(r), 'epoch')
    print(r[['epoch', 'train/box_loss', 'val/box_loss', 'metrics/precision(B)',
             'metrics/recall(B)', 'metrics/mAP50(B)']].tail(5))

## 6. Evaluasi

Yang menentukan adalah **MAE jumlah motor per foto** (bukan mAP). Urutannya: tuning `conf` di val (ROW6), lalu uji **sekali** di ROW7-8 dan jangan tuning lagi setelah melihat angkanya.

### 6.1 Tuning `conf` di val (ROW6)

Hasil: `conf` 0.4-0.5 memberi MAE 0.50 dan bias berbalik tanda di sekitar nol (area datar), jadi dipilih **0.45**. `conf` rendah kelebihan hitung, `conf` ≥0.6 kehilangan motor. `iou` 0.7-0.8 memperburuk (kotak ganda), tetap **0.6**.

In [ ]:
val_imgs = sorted(glob.glob(f'{DS}/images/val/*.jpg'))
gt_val = {}
for p in val_imgs:
    lbl = p.replace('images', 'labels')[:-4] + '.txt'
    gt_val[p] = sum(1 for _ in open(lbl)) if os.path.exists(lbl) else 0

res = []
for c in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    err = np.array([len(best(p, conf=c, iou=IOU, imgsz=IMGSZ, verbose=False)[0].boxes) - gt_val[p]
                    for p in val_imgs])
    res.append((c, np.abs(err).mean().round(2), err.mean().round(2)))
print(pd.DataFrame(res, columns=['conf', 'MAE', 'bias']))

### 6.2 Uji akhir di ROW7-8 (v3 vs baseline)

Hitung manual 16 foto (8 pagi + 8 malam) dengan aturan label yang sama, **tanpa melihat prediksi model dulu**. Hasil sebelumnya: v3 MAE 0.81 (pagi 0.62, malam 1.00), baseline MAE 1.00 (pagi 1.38, malam 0.62). Kegagalan v3 di malam terkonsentrasi di blok padat (ROW7A, ROW7B: motor depan berdempetan berskor rendah).

In [ ]:
sample_test = (d[d.row.isin(['ROW7', 'ROW8'])]
               .groupby('sesi').sample(8, random_state=7)[['sesi', 'file']]
               .sort_values(['sesi', 'file']))
sample_test['gt_depan'] = ''
sample_test.to_csv('gt_test.csv', index=False)
files.download('gt_test.csv')

In [ ]:
up = files.upload()                    # gt_test.csv yang sudah diisi
test = pd.read_csv('gt_test.csv')
assert test['gt_depan'].notna().all(), 'Masih ada gt_depan yang kosong'
test['gt_depan'] = test['gt_depan'].astype(float)

test['pred_v3'] = [len(best(os.path.join(DATA_DIR, r.sesi, r.file),
                            conf=CONF_FT, iou=IOU, imgsz=IMGSZ, verbose=False)[0].boxes)
                   for r in test.itertuples()]
test['pred_base'] = [hitung_base((r.sesi, r.file)) for r in test.itertuples()]

for k in ['pred_v3', 'pred_base']:
    e = test[k] - test['gt_depan']
    print(k, '| MAE:', e.abs().mean().round(2), '| bias:', e.mean().round(2),
          '|', test.assign(e=e).groupby('sesi')['e'].agg(lambda x: x.abs().mean()).round(2).to_dict())

test['err_v3'] = test['pred_v3'] - test['gt_depan']
print(test[['sesi', 'file', 'gt_depan', 'pred_v3', 'pred_base', 'err_v3']].to_string(index=False))

### 6.3 Cek visual foto bermasalah

Hijau = dihitung (skor ≥ `CONF_FT`), oranye = terdeteksi tapi tersaring. Berguna untuk melihat apakah motor depan terlewat (oranye / tanpa kotak) atau motor belakang ikut terhitung.

In [ ]:
def tampilkan(daftar, conf=0.2):
    # daftar: list (sesi, nama_file)
    fig, axs = plt.subplots(1, len(daftar), figsize=(9 * len(daftar), 7))
    axs = np.atleast_1d(axs)
    for ax, (sesi, nama) in zip(axs, daftar):
        p = os.path.join(DATA_DIR, sesi, nama)
        r = best(p, conf=conf, iou=IOU, imgsz=IMGSZ, verbose=False)[0]
        im = cv2.imread(p)
        for (x1, y1, x2, y2), c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()):
            col = (0, 255, 0) if c >= CONF_FT else (0, 165, 255)
            cv2.rectangle(im, (int(x1), int(y1)), (int(x2), int(y2)), col, 4)
            cv2.putText(im, f'{c:.2f}', (int(x1), int(y1) - 8), cv2.FONT_HERSHEY_SIMPLEX, 1.5, col, 4)
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.set_title(nama); ax.axis('off')
    plt.show()

tampilkan([('MALAM', 'ROW7A_MALAM.jpg'), ('MALAM', 'ROW7B_MALAM.jpg'), ('MALAM', 'ROW7C_MALAM.jpg')])

## 7. Output akhir: `hasil_v3.csv`

Hitung semua foto ROW dengan model v3, lalu tentukan status per blok:
- **penuh**: terisi ≥ kapasitas atau kosong ≤ 1
- **banyak kosong**: kosong ≥ 50% kapasitas
- **sebagian kosong**: sisanya

File ini dipakai `dashboard_parkiran.py` (kolom wajib: `sesi, row, titik, kapasitas, motor_depan_v3`). Foto SIDE dan nama bawaan HP tidak ikut karena belum punya kapasitas.

In [ ]:
d['motor_depan_v3'] = [len(best(os.path.join(DATA_DIR, r.sesi, r.file),
                                conf=CONF_FT, iou=IOU, imgsz=IMGSZ, verbose=False)[0].boxes)
                       for r in tqdm(list(d.itertuples()))]
d['slot_kosong'] = (d['kapasitas'] - d['motor_depan_v3']).clip(lower=0)
d['lokasi'] = d['row'] + d['titik']

def status(r):
    if r.motor_depan_v3 >= r.kapasitas or r.slot_kosong <= 1:
        return 'penuh'
    return 'banyak kosong' if r.slot_kosong / r.kapasitas >= 0.5 else 'sebagian kosong'
d['status'] = d.apply(status, axis=1)

d['waktu_update'] = datetime.now(ZoneInfo('Asia/Jakarta')).strftime('%Y-%m-%d %H:%M')
d.to_csv(f'{OUT}/hasil_v3.csv', index=False)

ringkas = d.groupby(['sesi', 'row'])[['kapasitas', 'motor_depan_v3', 'slot_kosong']].sum()
ringkas['okupansi_%'] = (100 * ringkas['motor_depan_v3'] / ringkas['kapasitas']).round(1)
print(ringkas)

In [ ]:
files.download(f'{OUT}/hasil_v3.csv')